In [1]:
import warnings

import numpy as np
import pandas as pd
import zlib

import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

from adaptive_behaviour import paths

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)

BASE_SEED = 20260817   # every bootstrap is seeded from this, so all tables are reproducible
N_BOOT = 100_000       # vectorised, so a large number of resamples is cheap

BIO_GROUPS = {0: "MCC", 2: "HCC"}
EXCLUDED_CHIP = 7282          # excluded in the published figures; retained here for consistency

AGENT_FILES = {
    "CFL-1": paths.RESULTS_CL / "data_clmethod_M1.csv",
    "CFL-2": paths.RESULTS_CL / "data_clmethod_M2.csv",
    "CFL-3": paths.RESULTS_CL / "data_clmethod_M3.csv",
    "CFL-4": paths.RESULTS_CL / "data_clmethod_M4.csv",
    "AIF-1": paths.RESULTS_SI / "p_data.csv",
    "DP-2":  paths.RESULTS_DPEFE / "p_data_2.csv",
    "DP-5":  paths.RESULTS_DPEFE / "p_data_5.csv",
    "DP-10": paths.RESULTS_DPEFE / "p_data_10.csv",
    "DP-20": paths.RESULTS_DPEFE / "p_data_20.csv",
}

METRICS = {
    "rally":      "Average rally length",   # higher is better
    "long_rally": "% long rallies",         # higher is better
    "ace":        "% aces",                 # LOWER is better
}
HIGHER_IS_BETTER = {"rally": True, "long_rally": True, "ace": False}

In [2]:
def load_biological():
    df = pd.read_csv(paths.require_in_vitro())
    df = df[(df.chip_id != EXCLUDED_CHIP) & (df.group.isin(BIO_GROUPS))].copy()
    df["group_name"]   = df.group.map(BIO_GROUPS)
    df["source"]       = "in-vitro"
    df["replicate_id"] = df.group_name + "|chip" + df.chip_id.astype(str)
    df["session_id"]   = (df.replicate_id + "|" + df.date.astype(str)
                          + "|s" + df.session_num.astype(str))
    return df[["source", "group_name", "replicate_id", "session_id", "half", "hit_count"]]


def load_agents():
    out = []
    for name, path in AGENT_FILES.items():
        d = pd.read_csv(path)
        d["group_name"]   = name
        d["source"]       = "in-silico"
        # one seed == one independent run == one session; there is no chip level in silico
        d["replicate_id"] = f"{name}|seed" + d.session_num.astype(int).astype(str)
        d["session_id"]   = d["replicate_id"]
        out.append(d[["source", "group_name", "replicate_id",
                      "session_id", "half", "hit_count"]])
    return pd.concat(out, ignore_index=True)


ep = pd.concat([load_biological(), load_agents()], ignore_index=True)
ep["half"]       = ep.half.astype(int)          # 0 = minutes 0-5, 1 = minutes 6-20
ep["long_rally"] = (ep.hit_count >= 3).astype(int) * 100
ep["ace"]        = (ep.hit_count == 0).astype(int) * 100
ep = ep.rename(columns={"hit_count": "rally"})

units = (ep.groupby(["source", "group_name"])
           .agg(episodes=("rally", "size"),
                sessions=("session_id", "nunique"),
                replicates=("replicate_id", "nunique"))
           .reset_index())
units["sessions_per_replicate"] = (units.sessions / units.replicates).round(1)
units

,source,group_name,episodes,sessions,replicates,sessions_per_replicate
0,in-silico,AIF-1,6930,99,99,1.0
1,in-silico,CFL-1,7000,100,100,1.0
2,in-silico,CFL-2,7000,100,100,1.0
3,in-silico,CFL-3,7000,100,100,1.0
4,in-silico,CFL-4,7000,100,100,1.0
5,in-silico,DP-10,6930,99,99,1.0
6,in-silico,DP-2,6930,99,99,1.0
7,in-silico,DP-20,6930,99,99,1.0
8,in-silico,DP-5,6930,99,99,1.0
9,in-vitro,HCC,9528,138,12,11.5


In [3]:
sess = (ep.groupby(["source", "group_name", "replicate_id", "session_id", "half"],
                   as_index=False)[list(METRICS)].mean())

wide = sess.pivot_table(index=["source", "group_name", "replicate_id", "session_id"],
                        columns="half", values=list(METRICS))
wide.columns = [f"{m}_h{h}" for m, h in wide.columns]
wide = wide.dropna().reset_index()

dropped = (wide.rally_h0 == 0).groupby(wide.group_name).sum()
print("Sessions removed by the zero-denominator guard:")
print(dropped[dropped > 0].to_string() if (dropped > 0).any() else "  none")
print(f"\nTotal removed: {int((wide.rally_h0 == 0).sum())} of {len(wide)} sessions")

wide = wide[wide.rally_h0 != 0].copy()
wide["RI"] = (wide.rally_h1 - wide.rally_h0) / wide.rally_h0 * 100

Sessions removed by the zero-denominator guard:
group_name
CFL-4    1

Total removed: 1 of 1143 sessions


In [4]:
def cluster_boot(values, clusters, label, n_boot=N_BOOT):
    """Bootstrap a mean, resampling whole clusters rather than individual rows.

    The mean of a concatenation of resampled clusters is the size-weighted mean of those
    clusters, so the whole bootstrap vectorises. The RNG is seeded deterministically from
    ``label``, which makes every table below reproducible and independent of the order in
    which cells are run.
    """
    rng = np.random.default_rng(BASE_SEED + zlib.crc32(label.encode()))
    df = pd.DataFrame({"v": values, "c": clusters})
    grouped = [g["v"].to_numpy() for _, g in df.groupby("c", sort=False)]
    sizes = np.array([len(g) for g in grouped])
    means = np.array([g.mean() for g in grouped])
    idx = rng.integers(0, len(grouped), (n_boot, len(grouped)))
    return (sizes[idx] * means[idx]).sum(axis=1) / sizes[idx].sum(axis=1)


rows = []
for gname, g in wide.groupby("group_name"):
    v = g.RI.to_numpy()
    se = v.std(ddof=1) / np.sqrt(len(v))
    naive_lo, naive_hi = v.mean() - 1.96 * se, v.mean() + 1.96 * se
    b = cluster_boot(v, g.replicate_id.to_numpy(), f"RI|{gname}")
    clo, chi = np.percentile(b, [2.5, 97.5])
    rows.append(dict(source=g.source.iloc[0], group=gname, n_sessions=len(v),
                     n_replicates=g.replicate_id.nunique(), mean_RI=v.mean(),
                     naive_width=naive_hi - naive_lo, cluster_lo=clo, cluster_hi=chi,
                     cluster_width=chi - clo))

ci_table = pd.DataFrame(rows)
ci_table["width_inflation"] = (ci_table.cluster_width / ci_table.naive_width).round(2)
ci_table.sort_values(["source", "group"]).round(2)

,source,group,n_sessions,n_replicates,mean_RI,naive_width,cluster_lo,cluster_hi,cluster_width,width_inflation
0,in-silico,AIF-1,99,99,15.81,23.25,4.67,27.82,23.14,1.00
1,in-silico,CFL-1,100,100,47.16,45.17,26.18,71.21,45.03,1.00
2,in-silico,CFL-2,100,100,40.71,35.58,23.85,59.24,35.40,0.99
3,in-silico,CFL-3,100,100,75.64,48.46,52.59,100.84,48.25,1.00
4,in-silico,CFL-4,99,99,106.22,58.86,78.47,136.78,58.31,0.99
5,in-silico,DP-10,99,99,66.94,41.25,47.14,88.31,41.17,1.00
6,in-silico,DP-2,99,99,49.05,41.96,29.14,70.80,41.66,0.99
7,in-silico,DP-20,99,99,39.97,34.14,23.61,57.54,33.93,0.99
8,in-silico,DP-5,99,99,70.94,61.29,43.36,104.13,60.76,0.99
9,in-vitro,HCC,138,12,57.14,22.33,37.65,82.00,44.35,1.99


In [5]:
within = []
for (src, gname), g in sess.groupby(["source", "group_name"]):
    for m in METRICS:
        fit = smf.mixedlm(f"{m} ~ half", data=g, groups=g["replicate_id"]).fit()
        lo, hi = fit.conf_int().loc["half"]
        within.append(dict(source=src, group=gname, metric=m,
                           n_replicates=g.replicate_id.nunique(),
                           delta=fit.params["half"], lo=lo, hi=hi, p=fit.pvalues["half"]))

within = pd.DataFrame(within)
within["p_adj"] = multipletests(within.p, method="fdr_bh")[1]
within["improved"] = np.where(
    within.p_adj >= 0.05, "n.s.",
    np.where(np.sign(within.delta) == np.where(within.metric.map(HIGHER_IS_BETTER), 1, -1),
             "improved", "worsened"))

for m, label in METRICS.items():
    print(f"\n--- {label} ({'higher' if HIGHER_IS_BETTER[m] else 'lower'} is better) ---")
    print(within[within.metric == m]
          .sort_values("delta", ascending=not HIGHER_IS_BETTER[m])
          .drop(columns="metric")
          .to_string(index=False, float_format=lambda x: f"{x:9.4f}"))


--- Average rally length (higher is better) ---
   source group  n_replicates     delta        lo        hi         p     p_adj improved
in-silico CFL-4           100    1.1584    0.4864    1.8303    0.0007    0.0024 improved
in-silico CFL-3           100    0.4502    0.2520    0.6483    0.0000    0.0000 improved
in-silico DP-10            99    0.3541    0.2068    0.5014    0.0000    0.0000 improved
in-silico  DP-5            99    0.3041    0.1554    0.4528    0.0001    0.0003 improved
 in-vitro   HCC            12    0.2431    0.1899    0.2963    0.0000    0.0000 improved
in-silico CFL-2           100    0.1761    0.0633    0.2890    0.0022    0.0067 improved
 in-vitro   MCC             7    0.1470    0.0898    0.2042    0.0000    0.0000 improved
in-silico CFL-1           100    0.1469   -0.0128    0.3067    0.0715    0.1388     n.s.
in-silico DP-20            99    0.1448    0.0248    0.2649    0.0181    0.0426 improved
in-silico  DP-2            99    0.1052   -0.0374    0.2478  

In [6]:
def contrast(bootA, bootB, valsA, valsB):
    d = bootA - bootB
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = max(2 * min((d <= 0).mean(), (d >= 0).mean()), 1 / N_BOOT)
    na, nb = len(valsA), len(valsB)
    sp = np.sqrt(((na - 1) * valsA.std(ddof=1) ** 2 + (nb - 1) * valsB.std(ddof=1) ** 2)
                 / (na + nb - 2))
    g = np.nan if sp == 0 else (valsA.mean() - valsB.mean()) / sp * (1 - 3 / (4 * (na + nb) - 9))
    return d.mean(), lo, hi, g, p


h1 = sess[sess.half == 1]
abs_vals = {g: x for g, x in h1.groupby("group_name")}
abs_boot = {g: cluster_boot(x.rally.to_numpy(), x.replicate_id.to_numpy(), f"rally|{g}")
            for g, x in abs_vals.items()}

rows = []
for agent in AGENT_FILES:
    for culture in ("MCC", "HCC"):
        d, lo, hi, g, p = contrast(abs_boot[agent], abs_boot[culture],
                                   abs_vals[agent].rally.to_numpy(),
                                   abs_vals[culture].rally.to_numpy())
        rows.append(dict(agent=agent, culture=culture, diff=d, lo=lo, hi=hi,
                         hedges_g=g, p=p))

abs_contrasts = pd.DataFrame(rows)
abs_contrasts["p_adj"] = multipletests(abs_contrasts.p, method="fdr_bh")[1]
abs_contrasts["outcome"] = np.where(
    abs_contrasts.p_adj >= 0.05, "not distinguishable",
    np.where(abs_contrasts["diff"] > 0, "agent higher", "agent lower"))

print(f"significant after correction: "
      f"{(abs_contrasts.p_adj < 0.05).sum()} of {len(abs_contrasts)}")
abs_contrasts.round(3)

significant after correction: 13 of 18


,agent,culture,diff,lo,hi,hedges_g,p,p_adj,outcome
0,CFL-1,MCC,0.169,0.039,0.336,0.330,0.007,0.013,agent higher
1,CFL-1,HCC,0.139,0.000,0.312,0.281,0.050,0.060,not distinguishable
2,CFL-2,MCC,0.161,0.057,0.268,0.451,0.002,0.005,agent higher
3,CFL-2,HCC,0.131,0.017,0.247,0.375,0.023,0.032,agent higher
4,CFL-3,MCC,0.588,0.401,0.790,0.864,0.000,0.000,agent higher
5,CFL-3,HCC,0.558,0.366,0.765,0.864,0.000,0.000,agent higher
6,CFL-4,MCC,1.619,0.943,2.737,0.472,0.000,0.000,agent higher
7,CFL-4,HCC,1.589,0.912,2.710,0.492,0.000,0.000,agent higher
8,AIF-1,MCC,-0.049,-0.112,0.003,-0.255,0.067,0.076,not distinguishable
9,AIF-1,HCC,-0.079,-0.155,-0.010,-0.422,0.023,0.032,agent lower


In [7]:
# bootstrap distributions of mean relative improvement, one per group
boots = {g: cluster_boot(x.RI.to_numpy(), x.replicate_id.to_numpy(), f"RI|{g}")
         for g, x in wide.groupby("group_name")}

# the resolution floor: can this measure tell the two cultures apart at all?
d_cult = boots["HCC"] - boots["MCC"]
lo_c, hi_c = np.percentile(d_cult, [2.5, 97.5])
pooled_sd = np.sqrt((wide[wide.group_name == "MCC"].RI.var(ddof=1)
                     + wide[wide.group_name == "HCC"].RI.var(ddof=1)) / 2)
print(f"HCC - MCC on relative improvement: {d_cult.mean():+.1f} points, "
      f"95% CI [{lo_c:.1f}, {hi_c:.1f}]")
print(f"pooled SD of culture relative improvement: {pooled_sd:.1f} points")
print("-> the two culture types are NOT distinguishable from each other on this measure.")

# equivalence margin: the gap between the two biological groups, rounded
EQUIV_MARGIN = 18.0
print(f"\nEquivalence margin for the test below: +/-{EQUIV_MARGIN:.0f} points "
      f"(approximately the MCC-HCC gap)")

HCC - MCC on relative improvement: +14.1 points, 95% CI [-20.9, 45.9]
pooled SD of culture relative improvement: 69.0 points
-> the two culture types are NOT distinguishable from each other on this measure.

Equivalence margin for the test below: +/-18 points (approximately the MCC-HCC gap)


In [8]:
rows = []
for agent in AGENT_FILES:
    for culture in ("MCC", "HCC"):
        d, lo, hi, g, p = contrast(boots[agent], boots[culture],
                                   wide[wide.group_name == agent].RI.to_numpy(),
                                   wide[wide.group_name == culture].RI.to_numpy())
        rows.append(dict(agent=agent, culture=culture, diff_RI=d, lo=lo, hi=hi,
                         hedges_g=g, p=p))

ri_contrasts = pd.DataFrame(rows)
ri_contrasts["p_adj"] = multipletests(ri_contrasts.p, method="fdr_bh")[1]

# three-way verdict: different / equivalent / inconclusive
different = ri_contrasts.p_adj < 0.05
equivalent = (ri_contrasts.lo > -EQUIV_MARGIN) & (ri_contrasts.hi < EQUIV_MARGIN)
ri_contrasts["outcome"] = np.where(
    different, np.where(ri_contrasts.diff_RI > 0, "agent higher", "agent lower"),
    np.where(equivalent, "equivalent", "inconclusive"))

print(ri_contrasts.outcome.value_counts().to_string())
ri_contrasts.round(3)

outcome
inconclusive    16
agent higher     1
agent lower      1


,agent,culture,diff_RI,lo,hi,hedges_g,p,p_adj,outcome
0,CFL-1,MCC,3.963,-31.569,35.580,0.083,0.778,0.875,inconclusive
1,CFL-1,HCC,-10.133,-42.092,21.192,-0.110,0.524,0.725,inconclusive
2,CFL-2,MCC,-2.477,-35.532,24.878,0.018,0.952,0.952,inconclusive
3,CFL-2,HCC,-16.574,-46.150,10.402,-0.210,0.240,0.481,inconclusive
4,CFL-3,MCC,32.435,-4.049,65.232,0.364,0.083,0.298,inconclusive
5,CFL-3,HCC,18.338,-15.114,50.668,0.194,0.268,0.482,inconclusive
6,CFL-4,MCC,62.973,23.269,100.412,0.580,0.001,0.007,agent higher
7,CFL-4,HCC,48.877,11.760,85.599,0.448,0.011,0.054,inconclusive
8,AIF-1,MCC,-27.414,-57.667,-4.693,-0.356,0.012,0.054,inconclusive
9,AIF-1,HCC,-41.510,-68.546,-18.177,-0.646,0.000,0.007,agent lower


In [9]:
dp = wide[wide.group_name.str.startswith("DP-")].copy()
dp["T"] = dp.group_name.str.split("-").str[1].astype(int)

for T in sorted(dp["T"].unique()):
    b = boots[f"DP-{T}"]
    sub = dp[dp["T"] == T]
    print(f"  DP-{T:<3d} n={len(sub):3d}  mean RI={sub.RI.mean():7.2f}"
          f"   95% CI [{np.percentile(b, 2.5):7.2f}, {np.percentile(b, 97.5):7.2f}]")

fit = smf.ols("RI ~ np.log(T)", data=dp).fit()
lo, hi = fit.conf_int().loc["np.log(T)"]
print(f"\n  OLS slope on log(T): {fit.params['np.log(T)']:+.3f}"
      f"  95% CI [{lo:.3f}, {hi:.3f}]  p = {fit.pvalues['np.log(T)']:.4f}")

  DP-2   n= 99  mean RI=  49.05   95% CI [  29.14,   70.80]
  DP-5   n= 99  mean RI=  70.94   95% CI [  43.36,  104.13]
  DP-10  n= 99  mean RI=  66.94   95% CI [  47.14,   88.31]
  DP-20  n= 99  mean RI=  39.97   95% CI [  23.61,   57.54]

  OLS slope on log(T): -3.140  95% CI [-16.658, 10.379]  p = 0.6482


In [10]:
tables = {}
for m, label in METRICS.items():
    rows = []
    for g, x in h1.groupby("group_name"):
        b = cluster_boot(x[m].to_numpy(), x.replicate_id.to_numpy(), f"{m}|{g}")
        rows.append(dict(group=g, mean=x[m].mean(),
                         lo=np.percentile(b, 2.5), hi=np.percentile(b, 97.5)))
    t = (pd.DataFrame(rows)
         .sort_values("mean", ascending=not HIGHER_IS_BETTER[m])
         .reset_index(drop=True))
    tables[m] = t
    print(f"\n--- {label} ({'higher' if HIGHER_IS_BETTER[m] else 'lower'} is better) ---")
    print(t.to_string(index=False, float_format=lambda x: f"{x:8.3f}"))


--- Average rally length (higher is better) ---
group     mean       lo       hi
CFL-4    2.470    1.798    3.586
CFL-3    1.439    1.257    1.636
 DP-5    1.189    1.063    1.329
DP-10    1.179    1.051    1.324
CFL-1    1.019    0.900    1.181
CFL-2    1.011    0.920    1.110
DP-20    0.980    0.888    1.082
 DP-2    0.957    0.864    1.069
  HCC    0.880    0.821    0.948
  MCC    0.847    0.810    0.906
AIF-1    0.801    0.767    0.836



--- % long rallies (higher is better) ---
group     mean       lo       hi
CFL-4   19.053   17.082   21.359
CFL-3   14.097   12.644   15.653
DP-20   11.696   10.735   12.671
DP-10   11.689   10.765   12.629
CFL-1   11.188   10.131   12.282
 DP-2   11.161   10.271   12.080
 DP-5   11.148   10.287   12.027
CFL-2   11.103    9.949   12.291
  MCC    9.226    8.695   10.488
AIF-1    8.468    7.659    9.320
  HCC    8.055    7.166    8.946



--- % aces (lower is better) ---
group     mean       lo       hi
CFL-4   47.025   44.748   49.246
  HCC   53.944   52.537   55.067
CFL-3   54.195   51.797   56.563
  MCC   54.260   52.354   55.553
AIF-1   54.265   52.793   55.744
CFL-1   56.747   54.815   58.649
CFL-2   57.087   54.665   59.508
DP-10   57.323   55.766   58.913
 DP-5   57.846   56.274   59.426
 DP-2   58.578   56.681   60.468
DP-20   59.701   57.697   61.695


In [11]:
def rank_of(metric, group):
    return int(tables[metric].index[tables[metric].group == group][0]) + 1


ace_mean = tables["ace"].set_index("group")["mean"]
n_sig_abs = int((abs_contrasts.p_adj < 0.05).sum())

claims = [
    ("CFL agents outperform MCC and HCC in all metrics (T <= 4)",
     "PARTLY SUPPORTED",
     f"True for rally length, where {n_sig_abs} of 18 contrasts are significant and CFL-2/3/4 "
     f"exceed HCC. Not true for aces: CFL-1 ({ace_mean['CFL-1']:.1f}) and CFL-2 "
     f"({ace_mean['CFL-2']:.1f}) are worse than MCC ({ace_mean['MCC']:.1f}) and "
     f"HCC ({ace_mean['HCC']:.1f})."),
    ("CFL-1 and CFL-2 perform comparably to MCC and HCC",
     "NOT SUPPORTED AS STATED",
     "On rally length both are significantly ABOVE MCC, and CFL-2 above HCC. On relative "
     "improvement the contrasts are inconclusive, not equivalent. The defensible claim is that "
     "short horizons land near the biological range (+0.14 to +0.17 hits) while long horizons "
     "leave it (CFL-4, +1.60)."),
    ("CFL-3 shows the strongest overall performance",
     "NOT SUPPORTED",
     f"CFL-4 ranks above CFL-3 on all three metrics "
     f"(rally {rank_of('rally','CFL-4')} vs {rank_of('rally','CFL-3')}, "
     f"long rallies {rank_of('long_rally','CFL-4')} vs {rank_of('long_rally','CFL-3')}, "
     f"aces {rank_of('ace','CFL-4')} vs {rank_of('ace','CFL-3')})."),
    ("CFL-4 exhibits the largest gain over time",
     "SUPPORTED",
     "Highest mean relative improvement and the largest absolute advantage over both cultures."),
    ("AIF-1 achieves only small gains over time",
     "SUPPORTED (in fact stronger)",
     "No detectable improvement in rally length at all (delta = -0.016, p_adj = 0.71), and "
     "significantly below HCC on absolute rally length."),
    ("Increasing the DPEFE planning horizon yields no substantial gain",
     "SUPPORTED",
     f"OLS slope on log(T) is {fit.params['np.log(T)']:+.2f}, p = {fit.pvalues['np.log(T)']:.2f}."),
    ("Performance declines above T = 10, an over-planning effect",
     "NOT SUPPORTED",
     "DP-20 and DP-10 intervals overlap and the horizon trend is not significant."),
    ("DP-T agents improve similarly to HCC",
     "NOT SUPPORTED AS STATED",
     "On relative improvement the contrasts are inconclusive rather than equivalent; on rally "
     "length DP-5 and DP-10 are significantly ABOVE both cultures."),
]

pd.set_option("display.max_colwidth", 95)
pd.DataFrame(claims, columns=["Claim in current Results", "Verdict", "Evidence"])

,Claim in current Results,Verdict,Evidence
0,CFL agents outperform MCC and HCC in all metrics (T <= 4),PARTLY SUPPORTED,"True for rally length, where 13 of 18 contrasts are significant and CFL-2/3/4 exceed HCC. N..."
1,CFL-1 and CFL-2 perform comparably to MCC and HCC,NOT SUPPORTED AS STATED,"On rally length both are significantly ABOVE MCC, and CFL-2 above HCC. On relative improvem..."
2,CFL-3 shows the strongest overall performance,NOT SUPPORTED,"CFL-4 ranks above CFL-3 on all three metrics (rally 1 vs 2, long rallies 1 vs 2, aces 1 vs 3)."
3,CFL-4 exhibits the largest gain over time,SUPPORTED,Highest mean relative improvement and the largest absolute advantage over both cultures.
4,AIF-1 achieves only small gains over time,SUPPORTED (in fact stronger),"No detectable improvement in rally length at all (delta = -0.016, p_adj = 0.71), and signif..."
5,Increasing the DPEFE planning horizon yields no substantial gain,SUPPORTED,"OLS slope on log(T) is -3.14, p = 0.65."
6,"Performance declines above T = 10, an over-planning effect",NOT SUPPORTED,DP-20 and DP-10 intervals overlap and the horizon trend is not significant.
7,DP-T agents improve similarly to HCC,NOT SUPPORTED AS STATED,On relative improvement the contrasts are inconclusive rather than equivalent; on rally len...


In [12]:
from pathlib import Path

TAB_DIR = paths.OUTPUTS_TABLES
TAB_DIR.mkdir(parents=True, exist_ok=True)


def fmt_p(p):
    return "$<$0.001" if p < 0.001 else f"{p:.3f}"


def ci(lo, hi, d=2):
    return f"[{lo:.{d}f}, {hi:.{d}f}]"


def booktabs(colspec, header, rows, caption, label, note=None, small=False):
    out = ["\\begin{table}[ht]", "\\centering"]
    if small:
        out.append("\\small")
    out += [f"\\caption{{{caption}}}", f"\\label{{{label}}}",
            f"\\begin{{tabular}}{{{colspec}}}", "\\toprule", header + " \\\\", "\\midrule"]
    out += [r + " \\\\" for r in rows]
    out += ["\\bottomrule", "\\end{tabular}"]
    if note:
        out.append(f"\\\\[4pt]\\footnotesize{{{note}}}")
    out.append("\\end{table}")
    return "\n".join(out) + "\n"


# ---- T2: sample structure -------------------------------------------------
u = units.set_index("group_name")
inflate = ci_table.set_index("group").width_inflation
rows_units = [
    f"In vitro & MCC & 7 chips & {int(u.loc['MCC','sessions'])} & {int(u.loc['MCC','episodes']):,}",
    f"In vitro & HCC & 12 chips & {int(u.loc['HCC','sessions'])} & {int(u.loc['HCC','episodes']):,}",
    "In silico & CFL-$T$ (each $T$) & 100 seeds & 100 & 7,000",
    "In silico & AIF-1 & 99 seeds & 99 & 6,930",
    "In silico & DP-$T$ (each $T$) & 99 seeds & 99 & 6,930",
]
(TAB_DIR / "tab_units.tex").write_text(booktabs(
    "llrrr", "Source & Group & Replicates & Sessions & Episodes", rows_units,
    "Units of analysis. Episode counts are matched by design between in-vitro and in-silico "
    "groups, but the biological sessions are nested within a small number of cultures.",
    "tab:units",
    "Replicates are the level at which observations are independent: the culture (chip) in vitro, "
    "the random seed in silico. Because many sessions are recorded from the same culture, "
    "treating sessions as independent understates biological variability; resampling whole chips "
    f"widens the 95\\% interval on relative improvement by {inflate['HCC']:.2f}$\\times$ for HCC and "
    f"{inflate['MCC']:.2f}$\\times$ for MCC, while leaving in-silico intervals unchanged."))

# ---- T1: agent vs culture, absolute rally length --------------------------
rows_abs = [f"{r.agent} & {r.culture} & {r['diff']:+.3f} & {ci(r.lo, r.hi, 3)} & "
            f"{r.hedges_g:+.2f} & {fmt_p(r.p_adj)} & {r.outcome}"
            for _, r in abs_contrasts.iterrows()]
(TAB_DIR / "tab_contrasts.tex").write_text(booktabs(
    "llrrrrl",
    "Agent & Culture & $\\Delta$ rally & 95\\% CI & Hedges' $g$ & $p_{\\mathrm{adj}}$ & Outcome",
    rows_abs,
    "Average rally length reached in the second window (6-20 min) by each in-silico agent, "
    "compared with the mouse (MCC) and human (HCC) cortical cultures. Differences and intervals "
    "come from a bootstrap that resamples whole replicates; $p$-values are Benjamini-Hochberg "
    "corrected across all 18 contrasts.",
    "tab:contrasts",
    "Positive $\\Delta$ favours the agent. ``Not distinguishable'' means the difference is not "
    "resolvable at this sample size, and is not evidence that the two groups match; see "
    "Supplementary Table \\mbox{\\ref{tab:contrastsri}} for the corresponding equivalence "
    "assessment.", small=True))

# ---- S1: within-group change ---------------------------------------------
rows_within = []
for g in ["MCC", "HCC"] + list(AGENT_FILES):
    cs = [g]
    for m in METRICS:
        r = within[(within.group == g) & (within.metric == m)].iloc[0]
        cs.append(f"{r.delta:+.3f} {ci(r.lo, r.hi, 3)} & {fmt_p(r.p_adj)}")
    rows_within.append(" & ".join(cs))
(TAB_DIR / "tab_within.tex").write_text(booktabs(
    "l" + "ll" * 3,
    "Group & \\multicolumn{2}{c}{$\\Delta$ rally length} & \\multicolumn{2}{c}{$\\Delta$ \\% long rallies} "
    "& \\multicolumn{2}{c}{$\\Delta$ \\% aces}",
    rows_within,
    "Change from the first (0--5 min) to the second (6--20 min) window, from a mixed-effects "
    "model with a random intercept on replicate. Each cell gives the estimate with its 95\\% "
    "confidence interval and the corrected $p$-value.",
    "tab:within",
    "Higher is better for rally length and long rallies; lower is better for aces. "
    "$p$-values are corrected across all 33 group $\\times$ metric tests.", small=True))

# ---- S2: level versus change ---------------------------------------------
lvl = (wide.groupby("group_name")
       .agg(w1=("rally_h0", "mean"), w2=("rally_h1", "mean"), RI=("RI", "mean"))
       .assign(gain=lambda d: d.w2 - d.w1)
       .sort_values("w2", ascending=False))
rows_level = [f"{g} & {r.w1:.3f} & {r.w2:.3f} & {r.gain:+.3f} & {r.RI:+.1f}"
              for g, r in lvl.iterrows()]
(TAB_DIR / "tab_level.tex").write_text(booktabs(
    "lrrrr",
    "Group & Window 1 & Window 2 & Absolute gain & Relative improvement (\\%)",
    rows_level,
    "Average rally length reached in each window, the absolute gain between them, and the "
    "relative improvement. Relative improvement normalises by each session's own first-window "
    "mean, so it rewards a low starting point.",
    "tab:level",
    "AIF-1 illustrates the limitation of the relative measure: its absolute change is negative "
    "($-0.016$) while its mean relative improvement is positive ($+15.8\\%$), because the mean "
    "of per-session ratios is not the ratio of means and is skewed by sessions with small "
    "first-window denominators."))

# ---- S3: absolute performance, all metrics -------------------------------
rows_absall = []
idx = {m: tables[m].set_index("group") for m in METRICS}
for g in ["MCC", "HCC"] + list(AGENT_FILES):
    cs = [g]
    for m in METRICS:
        r = idx[m].loc[g]
        cs.append(f"{r['mean']:.2f} {ci(r.lo, r.hi)}")
    rows_absall.append(" & ".join(cs))
(TAB_DIR / "tab_absolute.tex").write_text(booktabs(
    "llll", "Group & Rally length & \\% long rallies & \\% aces", rows_absall,
    "Absolute performance in the second window (6--20 min), with 95\\% intervals from a "
    "replicate-level bootstrap.",
    "tab:absolute",
    "Higher is better for rally length and long rallies; lower is better for aces. "
    "On aces, CFL-1 and CFL-2 are worse than both cultures.", small=True))

# ---- S4: agent vs culture, relative improvement --------------------------
rows_ri = [f"{r.agent} & {r.culture} & {r.diff_RI:+.1f} & {ci(r.lo, r.hi, 1)} & "
           f"{r.hedges_g:+.2f} & {fmt_p(r.p_adj)} & {r.outcome}"
           for _, r in ri_contrasts.iterrows()]
(TAB_DIR / "tab_contrasts_ri.tex").write_text(booktabs(
    "llrrrrl",
    "Agent & Culture & $\\Delta$RI (\\%) & 95\\% CI & Hedges' $g$ & $p_{\\mathrm{adj}}$ & Outcome",
    rows_ri,
    "Relative improvement of each in-silico agent against the two cultures, reported for "
    "continuity with the original DishBrain analysis. Outcomes are three-way: ``different'' "
    "where the contrast is significant after correction, ``equivalent'' where the interval falls "
    "entirely within a margin of $\\pm18$ percentage points (approximately the MCC--HCC gap), and "
    "``inconclusive'' otherwise.",
    "tab:contrastsri",
    f"No contrast reaches equivalence. The measure also fails to separate the two cultures from "
    f"each other (HCC $-$ MCC $= {d_cult.mean():+.1f}$ points, 95\\% CI "
    f"[{lo_c:.1f}, {hi_c:.1f}]), which sets the resolution floor for any comparison made with it.",
    small=True))

for f in sorted(TAB_DIR.glob("*.tex")):
    print(f"wrote {f.name:24s} {f.stat().st_size:5d} bytes")

wrote tab_absolute.tex          1271 bytes
wrote tab_contrasts.tex         2187 bytes
wrote tab_contrasts_ri.tex      2116 bytes
wrote tab_level.tex             1197 bytes
wrote tab_units.tex             1041 bytes
wrote tab_within.tex            1871 bytes
